# 04_nn_module

对应 `index.md` 第 4 阶段：`nn.Module`、常用层、`forward`。

笔记本在 `codes/pytorch/04_nn_module/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "04_nn_module" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "pytorch" / "04_nn_module"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/pytorch/04_nn_module')

## 1. 一层线性

用 `nn.Linear` 建一层：输入维度 `4`，输出维度 `2`。

对已有的 `x` 做一次前向，打印输出张量的 `shape`。


In [2]:
import torch
from torch import nn

x = torch.randn(3, 4)

# 作答

import torch

linear = nn.Linear(4, 2)
x = linear(x)
print(x.shape)
#评阅
# 对。Linear(4, 2)，输出 shape [3, 2]。

#参考答案
# layer = nn.Linear(4, 2)
# print(layer(x).shape)

torch.Size([3, 2])


/Users/keyficller/Documents/AEFS-Notes/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## 2. 自定义 Module

写一个继承 `nn.Module` 的类 `DoubleLinear`：

- 在 `__init__` 里注册一层 `nn.Linear(3, 3)`，名字自定
- 在 `forward` 里对输入做这层线性变换，再乘以 `2` 返回

实例化后对 `x` 前向，打印输出的 `shape`。


In [3]:
import torch
from torch import nn

x = torch.randn(2, 3)

# 作答

import torch
import torch.nn as nn


class DoubleLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 3)
    
    def forward(self, x):
        x = self.linear(x)
        return x * 2
    
    
linear = DoubleLinear()
x = linear(x)
print(x.shape)
#评阅
# 对。继承 Module、注册 Linear、forward 再乘 2，shape [2, 3]。

#参考答案
# class DoubleLinear(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.fc = nn.Linear(3, 3)
#     def forward(self, x):
#         return self.fc(x) * 2
# print(DoubleLinear()(x).shape)


torch.Size([2, 3])


## 3. Sequential 串两层

用 `nn.Sequential` 串起来：`Linear(5, 8)` → `ReLU` → `Linear(8, 1)`。

对 `x` 前向，打印输出的 `shape`。


In [4]:
import torch
from torch import nn

x = torch.randn(4, 5)

# 作答

import torch
import torch.nn as nn

seq = nn.Sequential(
    nn.Linear(5, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

x = seq(x)
print(x.shape)
#评阅
# 对。顺序与维度都对，输出 [4, 1]。

#参考答案
# net = nn.Sequential(nn.Linear(5, 8), nn.ReLU(), nn.Linear(8, 1))
# print(net(x).shape)

torch.Size([4, 1])


## 4. 看参数

已有模块 `net`。打印它包含多少个**可训练参数张量**（`parameters()` 里有多少个），再打印第一个参数张量的 `shape`。


In [5]:
import torch
from torch import nn

net = nn.Linear(6, 2)

# 作答

params = list(net.parameters())
print(len(params))
print(params[0].shape)
#评阅
# 对。Linear 有 weight+bias 共 2 个；第一个通常是 weight，shape [2, 6]。

#参考答案
# params = list(net.parameters())
# print(len(params))
# print(params[0].shape)


2
torch.Size([2, 6])


## 5. train / eval 模式

已有带 Dropout 的 `net`。先切到 `eval`，对同一份 `x` 前向两次，打印两次输出是否 `allclose`。

再切回 `train`，同样前向两次，打印两次输出是否 `allclose`（Dropout 开启时通常会不同）。


In [6]:
import torch
from torch import nn

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(4, 4), nn.Dropout(p=0.5), nn.Linear(4, 2))
x = torch.randn(2, 4)

# 作答

import torch

net.eval()
print(torch.allclose(net(x), net(x)))

net.train()
print(torch.allclose(net(x), net(x)))
#评阅
# 对。eval 两次一致 True；train 下 Dropout 打开，通常 False。

#参考答案
# net.eval()
# print(torch.allclose(net(x), net(x)))
# net.train()
# print(torch.allclose(net(x), net(x)))


True
False
